### 1. Setup and Imports
Loading the required Python libraries to process and visualize the C3D biomechanics data.

In [1]:
import ezc3d
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

### 2. File Setup
Defining the target directory and generating a list of all `.c3d` files available for processing.
Defining the output folder to store the relabeled files safely.

In [7]:
data_folder = 'data'  
output_folder = 'data_relabeled'
translation_file = 'translation_table.xlsx'

os.makedirs(output_folder, exist_ok=True) # Create the output directory if it doesn't exist
files = [f for f in os.listdir(data_folder) if f.endswith('.c3d')]  # Get list of all .c3d files

print(f"Found {len(files)} C3D files to process.")

Found 4 C3D files to process.


## 3. Analyzing the labels of the translation_table
Reading the `translation_table` containing our marker mappings.
* Column 1 contains the Model (Reference) Labels.
* Columns 2 to 5 contain the different Databases Labels.

In [32]:
df_translation = pd.read_excel(translation_file) # Read the excel file

# Extract column names based on their order
ref_name = df_translation.columns[0]
db_names = df_translation.columns[1:5]

db_mappings = {db: {} for db in db_names}  # Dictionary format: { 'DB_Name': { 'old_label': 'Ref_Label' } }

for index, row in df_translation.iterrows():
    ref_label = row[ref_name] # Grabs reference label    
    if pd.isna(ref_label):   # Skip if there's no reference label for this row
        continue
        
    for i, db_col in enumerate(db_names):
        old_label = row[db_col]
        if pd.notna(old_label):   # If the database has a label for this row
            clean_old = str(old_label).strip()
            db_mappings[db_col][clean_old] = str(ref_label).strip()  # Assign the reference label to the specific database's dictionary

print(f"Loaded {df_translation[ref_name].dropna().count()} total reference markers.")

Loaded 66 total reference markers.


## 4. Relabeling (aka retargeting)
Looping through each C3D file and checking its current markers against our label_mapping. 
If a marker is found, we swap it for the reference label. If it isn't found, we leave it as-is. Finally, we print out a summary for each file indicating which database it matched with and the ratio of relabeled vs. untouched markers.

In [45]:
processed_data = []  # List to temporarily store our modified c3d objects before saving

for file_name in files:
    file_path = os.path.join(data_folder, file_name)
    c3d_data = ezc3d.c3d(file_path)
    current_labels = c3d_data['parameters']['POINT']['LABELS']['value']
    
    clean_current_labels = [label.strip() for label in current_labels] # Clean all current labels upfront to avoid doing it repeatedly
    
    # 1. SCORING PHASE: Identify which database this file belongs to
    db_hits = {db: 0 for db in db_names}
    for label in clean_current_labels:
        for db_name, mapping in db_mappings.items():
            if label in mapping:
                db_hits[db_name] += 1
                
    primary_db = max(db_hits, key=db_hits.get)  # Find the database with the most hits
    max_hits = db_hits[primary_db]
    
    # 2. RELABELING PHASE
    new_labels = []
    relabeled_count = 0
    not_relabeled_count = 0
    
    if max_hits > 0:
        winning_mapping = db_mappings[primary_db]
        
        for clean_label in clean_current_labels:
            if clean_label in winning_mapping:
                new_labels.append(winning_mapping[clean_label])
                relabeled_count += 1
            else:
                new_labels.append(clean_label)
                not_relabeled_count += 1
                
        # Update labels in the C3D object
        c3d_data['parameters']['POINT']['LABELS']['value'] = new_labels
        
        if relabeled_count > 0: # Only save to memory if we actually changed something
            processed_data.append((file_name, c3d_data)) 
        print(f"File '{file_name}' ({primary_db}) -> {relabeled_count} markers were relabeled, while {not_relabeled_count} markers were not.") 
    else:
        print(f"File '{file_name}' -> 0 markers were relabeled, while {len(clean_current_labels)} markers were not.")

File 'P01+Ideal+LG+0+1+1.c3d' (DB4_Jiwei) -> 22 markers were relabeled, while 80 markers were not.
File 'P1 pw 01.c3d' (DB2_Wang) -> 30 markers were relabeled, while 8 markers were not.
File 'Ref01_sss_le_shoes04.c3d' (DB3_Vielemeyer) -> 17 markers were relabeled, while 180 markers were not.
File 'Walk_Comfortable_raw.c3d' -> 0 markers were relabeled, while 78 markers were not.


## 5. Exporting data after retarget
Iterating through our updated C3D objects and write them to our output directory `data_relabeled`.

In [46]:
for file_name, c3d_data in processed_data:
    # Force the labels into a tuple of standard strings for the C++ backend
    current_labels = c3d_data['parameters']['POINT']['LABELS']['value']
    c3d_data['parameters']['POINT']['LABELS']['value'] = tuple(str(lbl) for lbl in current_labels)
    
    new_file_name = file_name.replace('.c3d', '_relabeled.c3d')  # Rename the file 
    output_path = os.path.join(output_folder, new_file_name)
    
    c3d_data.write(output_path)   # Write the C3D file to disk
    print(f"Exported: {new_file_name}")

print("\nProcess complete! All files saved to the output folder.")

Exported: P01+Ideal+LG+0+1+1_relabeled.c3d
Exported: P1 pw 01_relabeled.c3d
Exported: Ref01_sss_le_shoes04_relabeled.c3d

Process complete! All files saved to the output folder.
